# RL-based Search for FlexTok Tokens

This notebook demonstrates how to use reinforcement learning agents to search over the FlexTok token space.

The RL agent learns to sequentially pick token values that minimize perceptual similarity to a target image.

In [ ]:
# Setup
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '5'

import sys
sys.path.append('..')

%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from flextok.utils.misc import detect_bf16_support
from twenty_questions.twenty_questions import load_flextok_model, convert_images_to_pil
from flextok.utils.dataloader import CelebAHQDataset, create_celeb_dataloader

# RL imports
from rl_search.environment import FlexTokSearchEnv
from rl_search.reward_shaping import RewardShaper, create_reward_shaper
from rl_search.agents import create_agent
from rl_search.eval_utils import (
    evaluate_agent, 
    visualize_episode, 
    compare_agents,
    visualize_search_trajectory,
    analyze_token_distribution,
)

# Automatically set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

# Detect if bf16 is enabled
enable_bf16 = detect_bf16_support()
print('BF16 enabled:', enable_bf16)

## 1. Load FlexTok Model and Target Image

In [ ]:
# Load FlexTok model
CKPT_PATH = "/home/iyu/ml-flextok/checkpoints/celeba_d18_arcface_fsq_8/20260108/checkpoint_best.pt"
FSQ_LEVELS = [8]

model = load_flextok_model(
    ckpt_path=CKPT_PATH,
    fsq_level=FSQ_LEVELS
).to(device)

print(f"Model loaded with FSQ levels: {FSQ_LEVELS}")

In [ ]:
# Load target image from dataset
IMG_SIZE = 256
DATASET_NAME = "celebahq"
DATASET_PATH = f"../data/{DATASET_NAME}/"

val_dataset = CelebAHQDataset(
    root_dir=DATASET_PATH,
    img_size=IMG_SIZE,
    split="val",
)

# Get a random target image
idx = np.random.randint(0, len(val_dataset))
target_tensor = val_dataset[idx].unsqueeze(0).to(device)
target_image = convert_images_to_pil(target_tensor)[0]

print(f"Loaded target image (index {idx})")
plt.figure(figsize=(4, 4))
plt.imshow(target_image)
plt.title("Target Image")
plt.axis('off')
plt.show()

## 2. Load Similarity Model (DreamSim)

In [ ]:
from dreamsim import dreamsim

# Load DreamSim model
dreamsim_model, dreamsim_preprocess = dreamsim(pretrained=True, device=device)

print("DreamSim model loaded")

In [ ]:
# Create similarity function
def similarity_fn(img1: Image.Image, img2: Image.Image) -> float:
    """Compute DreamSim similarity score between two PIL images."""
    img1_tensor = dreamsim_preprocess(img1).unsqueeze(0).to(device)
    img2_tensor = dreamsim_preprocess(img2).unsqueeze(0).to(device)
    
    with torch.no_grad():
        score = dreamsim_model(img1_tensor, img2_tensor)
    
    return float(score.item())

## 3. Create RL Environment

In [ ]:
# Create reward shaper
reward_config = {
    'type': 'simple',
    'mode': 'improvement',  # Options: 'negative', 'inverse', 'exponential', 'improvement'
}
reward_shaper = create_reward_shaper(reward_config)

# Create environment
env = FlexTokSearchEnv(
    flextok_model=model,
    target_image=target_image,
    similarity_fn=similarity_fn,
    fsq_levels=FSQ_LEVELS,
    max_tokens=256,  # Maximum number of tokens to predict
    history_length=5,  # Number of past (token, reward) to include in obs
    enable_bf16=enable_bf16,
    device=device,
    goal_threshold=0.1,  # Terminate early if score < 0.1
    enable_undo=True,  # Allow undo actions
    image_obs=False,  # Don't include images in observations (faster)
    reward_shaper=reward_shaper,
)

print("Environment created")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

## 4. Create and Train RL Agent

We'll create a DQN agent (best for discrete action spaces like FSQ [8]).

In [ ]:
# Create DQN agent
agent_config = {
    'learning_rate': 1e-4,
    'buffer_size': 50000,
    'batch_size': 32,
    'gamma': 0.99,
    'exploration_fraction': 0.3,
    'exploration_initial_eps': 1.0,
    'exploration_final_eps': 0.05,
    'use_custom_feature_extractor': True,
    'features_dim': 256,
    'verbose': 1,
}

agent = create_agent('dqn', env, agent_config)

print("DQN agent created")

In [ ]:
# Train the agent
TOTAL_TIMESTEPS = 50000  # Adjust based on your needs

print(f"Training agent for {TOTAL_TIMESTEPS} timesteps...")
agent.learn(total_timesteps=TOTAL_TIMESTEPS, log_interval=10)
print("Training complete!")

## 5. Evaluate the Trained Agent

In [ ]:
# Evaluate agent
eval_results = evaluate_agent(
    agent=agent,
    env=env,
    n_episodes=10,
    deterministic=True,
    verbose=True,
)

print("\n" + "="*50)
print("Evaluation Summary:")
print(f"Mean episode reward: {eval_results['mean_reward']:.2f} ± {eval_results['std_reward']:.2f}")
print(f"Mean best score: {eval_results['mean_best_score']:.4f}")
print(f"Mean final score: {eval_results['mean_final_score']:.4f}")
print("="*50)

## 6. Visualize Episode

In [ ]:
# Visualize a single episode
fig = visualize_episode(
    agent=agent,
    env=env,
    deterministic=True,
)
plt.show()

## 7. Visualize Search Trajectory

In [ ]:
# Get a trajectory from evaluation
trajectory = eval_results['trajectories'][0]

# Visualize trajectory
fig = visualize_search_trajectory(
    trajectory=trajectory,
    target_image=target_image,
)
plt.show()

## 8. Analyze Token Distribution

In [ ]:
# Analyze token choices
fig = analyze_token_distribution(
    trajectories=eval_results['trajectories'],
    fsq_levels=FSQ_LEVELS,
)
plt.show()

## 9. Compare with Greedy/Beam Search (Optional)

Let's compare the RL agent with the greedy/beam search from the original notebook.

In [ ]:
from twenty_questions.twenty_questions import auto_twenty_q, get_possible_combos, zhat_to_tokens

# Prepare tokens list for greedy search
all_zhats = get_possible_combos(model).to(device)
tokens_list = zhat_to_tokens(model, all_zhats).unsqueeze(-1)
tokens_list[-1] = torch.tensor([[FSQ_LEVELS[0] - 1]], device=tokens_list[-1].device)
tokens_list = list(tokens_list.split(1))

# Run greedy search
print("Running greedy search...")
chosen_history_greedy, _ = auto_twenty_q(
    flextok_model=model,
    secret_image=target_image,
    tokens_list=tokens_list,
    num_samples_per_quantization=4,
    enable_bf16=enable_bf16,
    eval_model=dreamsim_model,
    num_questions=256,
    search_algorithm="greedy",
    preprocess_fn=dreamsim_preprocess
)

greedy_scores = [float(item[2]) for item in chosen_history_greedy]
print(f"Greedy search best score: {min(greedy_scores):.4f}")

In [ ]:
# Compare RL vs Greedy
plt.figure(figsize=(12, 5))

# Plot RL trajectory
plt.subplot(1, 2, 1)
rl_scores = [step['score'] for step in eval_results['trajectories'][0]]
plt.plot(rl_scores, marker='o', markersize=3, label='RL Agent', color='blue')
plt.xlabel('Step')
plt.ylabel('Similarity Score')
plt.title('RL Agent Search')
plt.grid(alpha=0.3)
plt.legend()

# Plot Greedy trajectory
plt.subplot(1, 2, 2)
plt.plot(greedy_scores, marker='o', markersize=3, label='Greedy Search', color='green')
plt.xlabel('Step')
plt.ylabel('Similarity Score')
plt.title('Greedy Search')
plt.grid(alpha=0.3)
plt.legend()

plt.suptitle(f'RL (best: {min(rl_scores):.4f}) vs Greedy (best: {min(greedy_scores):.4f})')
plt.tight_layout()
plt.show()

## 10. Save and Load Agent

In [ ]:
# Save agent
save_path = "../rl_checkpoints/dqn_flextok_search.zip"
agent.save(save_path)
print(f"Agent saved to {save_path}")

In [ ]:
# Load agent
from rl_search.agents import DQNAgent

loaded_agent = DQNAgent(env)
loaded_agent.load(save_path)
print(f"Agent loaded from {save_path}")

## 11. Try Different Reward Shaping Strategies

In [ ]:
# Compare different reward shaping modes
reward_modes = ['negative', 'inverse', 'exponential', 'improvement']

for mode in reward_modes:
    print(f"\nTesting reward mode: {mode}")
    
    # Create environment with this reward mode
    test_reward_shaper = RewardShaper(mode=mode)
    test_env = FlexTokSearchEnv(
        flextok_model=model,
        target_image=target_image,
        similarity_fn=similarity_fn,
        fsq_levels=FSQ_LEVELS,
        max_tokens=256,
        enable_bf16=enable_bf16,
        device=device,
        reward_shaper=test_reward_shaper,
    )
    
    # Quick test
    obs, info = test_env.reset()
    total_reward = 0
    
    for _ in range(10):
        action = test_env.action_space.sample()
        obs, reward, terminated, truncated, info = test_env.step(action)
        total_reward += reward
        if terminated or truncated:
            break
    
    print(f"  Sample total reward: {total_reward:.2f}")
    print(f"  Final score: {info.get('current_score', 0):.4f}")

## Summary

This notebook demonstrated:

1. **Environment Setup**: Created a Gym-compatible environment for FlexTok token search
2. **Reward Shaping**: Explored different reward transformation strategies
3. **Agent Training**: Trained a DQN agent using Stable-Baselines3
4. **Evaluation**: Evaluated the agent's performance with comprehensive metrics
5. **Visualization**: Visualized search trajectories, token distributions, and comparisons

### Next Steps

- Try SAC agent for comparison (better for exploration)
- Experiment with multi-objective reward shaping
- Train on multiple target images to learn a general policy
- Tune hyperparameters for better performance
- Compare with beam search and other classical search algorithms